# Урок 13 - Пам’ять агента з Cognee Knowledge Graphs


## Налаштування

Цей нотатник демонструє, як створити інтелектуального **асистента з кодування** з постійною пам’яттю, використовуючи знання у вигляді графів від [**Cognee**](https://www.cognee.ai/) та **Microsoft Agent Framework** (MAF).

Cognee перетворює неструктурований текст у структурований, запитуваний граф знань, підкріплений векторними вбудовуваннями — надаючи вашому агенту багату довготривалу пам’ять, що враховує взаємозв’язки.

### Чому ви навчитесь
1. **Створювати графи знань**: перетворювати профілі розробників і найкращі практики на структуровані, запитувані знання.
2. **Інтегрувати Cognee з MAF**: використовувати функції `@tool`, щоб агент MAF міг робити запити до графа знань Cognee.
3. **Підтримувати контекст сесії**: зберігати контекст під час кількох питань у тій самій сесії.
4. **Довготривала пам’ять**: зберігати важливі знання між сесіями та витягувати їх у нових розмовах.

### Необхідні умови
- Python 3.9+
- Redis, що працює локально (`docker run -d -p 6379:6379 redis`), для керування сесіями
- Ключ API LLM (наприклад, OpenAI) — встановіть `LLM_API_KEY` у `.env`
- `CACHING=true` у `.env` (потрібно для сесій Cognee)
- Проект Microsoft Foundry із розгорнутою моделлю чату
- `AZURE_AI_PROJECT_ENDPOINT` і `AZURE_AI_MODEL_DEPLOYMENT_NAME` у `.env`
- Аутентифікація через Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Типи пам'яті агента

Цей запис досліджує ті самі три типи пам'яті з основного нотатника Урок 13, але використовує Cognee як бекенд для довготривалої пам'яті:

| Тип пам'яті | Механізм | Тривалість |
|---|---|---|
| **Робоча** | `agent.create_session()` (MAF) | Одна розмова |
| **Короткочасна** | Кеш сесії Cognee (Redis) | Одна сесія |
| **Довготривала** | Граф знань Cognee + вектори | Постійна |

### Архітектура пам'яті Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Підготуйте Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Частина 1 — Створення бази знань

Ми збираємо три типи даних для створення всебічної бази знань для нашого кодувального помічника:

1. **Профіль розробника** — особисті знання та технічний досвід
2. **Кращі практики Python** — The Zen of Python з практичними рекомендаціями
3. **Історичні розмови** — минулі сесії запитань і відповідей між розробниками та AI-помічниками


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Візуалізація графа знань

Cognee може створювати інтерактивну HTML-візуалізацію сутностей та їхніх зв’язків, які він витягнув.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Збагачення пам’яті за допомогою Memify

`memify()` аналізує граф знань і генерує інтелектуальні правила — визначаючи шаблони, найкращі практики та взаємозв’язки між поняттями.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Частина 2 — Агент MAF з інструментами Cognee

Тепер ми створюємо агента MAF, який може виконувати запити до графа знань Cognee через функції `@tool`. Це дозволяє агенту використовувати всю потужність семантичного пошуку, орієнтованого на граф, одночасно підтримуючи контекст розмови через сесії.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Робоча пам’ять з сесіями

`AgentSession` (створена через `agent.create_session()`) забезпечує робочу пам’ять у межах сесії. Агент може посилатися на попередні повідомлення, одночасно звертаючись до довготривалої графової бази знань Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Нова сесія — Довготривала пам’ять зберігається

Початок нової сесії очищує оперативну пам’ять, але граф знань Cognee все ще доступний. Агенти можуть отримати той самий довготривалий обсяг знань у цілком новій розмові.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

У цьому блокноті ви створили помічника з кодування, який поєднує **оперативну пам'ять MAF** (`agent.create_session()`) з **довготривалою графом знань Cognee**.

### Чому Ви Навчилися
1. **Побудова графа знань**: Cognee обробляє неструктурований текст і створює граф + векторну пам'ять.
2. **Збагачення графа за допомогою memify**: Виведені факти та більш багаті відносини поверх вашого існуючого графа.
3. **Інтеграція MAF + Cognee**: Функції `@tool` дозволяють агенту MAF природно здійснювати запити до графа Cognee.
4. **Оперативна пам'ять + довготривала пам'ять**: `AgentSession` (через `agent.create_session()`) забезпечує контекст сесії, тоді як Cognee надає стійкі знання.
5. **Фільтрований пошук за допомогою NodeSets**: Орієнтуйтеся на певні підмножини графа знань (наприклад, лише принципи).

### Основні Висновки
- **Cognee** перетворює сирий текст у структуровану, свідому відносинам пам'ять — потужнішу за просте векторне сховище.
- **Функції `@tool`** чисто з'єднують агентів MAF з зовнішніми системами знань.
- **`AgentSession`** (через `agent.create_session()`) зберігає контекст на рівні розмови окремо від довготривалих знань.
- Той самий граф знань обслуговує кілька сесій та агентів.

### Реальні Застосування
- **Розробницькі копілоти**: Перевірка коду, аналіз інцидентів, архітектурні помічники
- **Копілоти для клієнтів**: Агент підтримки, що працює з документацією, ЧЗВ та нотатками CRM
- **Внутрішні експертні копілоти**: Помічники з політик, юридичних або безпекових питань, що працюють з керівництвом
- **Уніфіковані шари даних**: Поєднання структурованих і неструктурованих даних в один запитуваний граф

### Подальші Кроки
- Експериментуйте з часовою обізнаністю в Cognee
- Визначте онтологію OWL для доменно-специфічної якості графа
- Додайте цикли зворотного зв’язку від користувачів для покращення пошуку з часом
- Масштабуйте до мультиагентних систем, які ділять той самий шар пам'яті Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
